In [ ]:

import pandas as pd

# 加载训练数据
train_data_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/reservation_cancellation/train.csv'
train_df = pd.read_csv(train_data_path)

# 加载测试数据
test_data_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/reservation_cancellation/test.csv'
test_df = pd.read_csv(test_data_path)

# 查看数据的部分信息
train_df.head(), test_df.head()


(      id  no_of_adults  ...  no_of_special_requests  booking_status
 0  15559             2  ...                       2               0
 1  32783             2  ...                       1               0
 2  11797             3  ...                       0               1
 3  39750             2  ...                       1               1
 4  28711             2  ...                       0               1
 
 [5 rows x 19 columns],
       id  no_of_adults  ...  no_of_special_requests  booking_status
 0   8768             2  ...                       1               0
 1  38340             2  ...                       0               1
 2   7104             2  ...                       0               0
 3  36898             2  ...                       3               0
 4   9747             2  ...                       1               0
 
 [5 rows x 19 columns])

In [ ]:
import numpy as np
from sklearn.preprocessing import LabelEncoder

# 检查数据类型和缺失值
train_df.info()
test_df.info()

# 对非数值特征进行编码
non_numeric_columns = train_df.select_dtypes(exclude=np.number).columns.tolist()

label_encoders = {}

for column in non_numeric_columns:
    le = LabelEncoder()
    train_df[column] = le.fit_transform(train_df[column].astype(str))
    test_df[column] = le.transform(test_df[column].astype(str))
    label_encoders[column] = le

train_df.head(), test_df.head()

Output truncated. Showing the last 2800 characters. You should try again and use computer.ai.summarize(output) over the output, or break it down into smaller steps.

ons          33674 non-null  int64  
 15  no_of_previous_bookings_not_canceled  33674 non-null  int64  
 16  avg_price_per_room                    33674 non-null  float64
 17  no_of_special_requests                33674 non-null  int64  
 18  booking_status                        33674 non-null  int64  
dtypes: float64(1), int64(18)
memory usage: 4.9 MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8420 entries, 0 to 8419
Data columns (total 19 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   id                                    8420 non-null   int64  
 1   no_of_adults                          8420 non-null   int64  
 2   no_of_children                        8420 non-null   int64  
 3   no_of_weekend_nights           

In [ ]:

# 分离特征和标签
X_train = train_df.drop(columns=['id', 'booking_status'])
y_train = train_df['booking_status']

X_test = test_df.drop(columns=['id', 'booking_status'])
y_test = test_df['booking_status']

# 输出特征和标签的形状
X_train.shape, y_train.shape, X_test.shape, y_test.shape


((33674, 17), (33674,), (8420, 17), (8420,))

In [ ]:

import lightgbm as lgb
from sklearn.metrics import roc_auc_score
import numpy as np

# 构建LightGBM数据集
train_data = lgb.Dataset(X_train, label=y_train)

# 设置LightGBM参数
params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'num_leaves': 31,
    'learning_rate': 0.05,
    'feature_fraction': 0.9,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': 0
}

# 训练模型
num_round = 100
bst = lgb.train(params, train_data, num_round)

# 进行预测
y_pred = bst.predict(X_test)

# 计算AUC-ROC得分
auc_roc = roc_auc_score(y_test, y_pred)
auc_roc

# 保存预测结果
predictions = np.where(y_pred > 0.5, 1, 0)
np.savetxt('predictions.csv', predictions, delimiter=',', fmt='%d')


In [ ]:

import lightgbm as lgb
from sklearn.metrics import roc_auc_score
import numpy as np

# 重新加载模型和数据
bst = lgb.Booster(model_file='lightgbm_model.txt')  # 如果保存了模型文件
y_pred = bst.predict(X_test)

# 计算AUC-ROC得分
auc_roc = roc_auc_score(y_test, y_pred)
auc_roc


---------------------------------------------------------------------------
LightGBMError                             Traceback (most recent call last)
Cell In[6], line 12
     10 pass
     11 print('##active_line6##')
---> 12 bst = lgb.Booster(model_file='lightgbm_model.txt')
     13 print('##active_line7##')
     14 y_pred = bst.predict(X_test)

File D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\lib\site-packages\lightgbm\basic.py:3678, in Booster.__init__(self, params, train_set, model_file, model_str)
   3675 elif model_file is not None:
   3676     # Prediction task
   3677     out_num_iterations = ctypes.c_int(0)
-> 3678     _safe_call(
   3679         _LIB.LGBM_BoosterCreateFromModelfile(
   3680             _c_str(str(model_file)),
   3681             ctypes.byref(out_num_iterations),
   3682             ctypes.byref(self._handle),
   3683         )
   3684     )
   3685     out_num_class = ctypes.c_int(0)
   3686     _safe_call(
   3687         _LIB.LGBM_Boo

In [ ]:


# 使用已经训练好的模型进行预测
y_pred = bst.predict(X_test)

# 计算AUC-ROC得分
auc_roc = roc_auc_score(y_test, y_pred)
auc_roc



np.float64(0.8935701434042931)